# Lab 1 — Concepts, environnement & Function Calling

## Objectif du notebook

Ce notebook explique, étape par étape, comment fonctionne un agent IA.

On va apprendre :

1. Ce qu'est un appel LLM.
2. Le rôle des messages `system`, `user`, `assistant`.
3. Le rôle de la température.
4. Comment lire les tokens et le coût.
5. Comment fonctionne le `function calling`.
6. Comment un agent utilise un outil.
7. Comment produire une sortie structurée en JSON.
8. Comment décrire un agent avec la méthode PEAS.

---

## Explication simple

Imagine un agent IA comme un petit robot intelligent.

Tu lui donnes une mission.

Il peut :

- écouter ta demande ;
- réfléchir ;
- décider s'il a besoin d'un outil ;
- utiliser l'outil ;
- te répondre.

C'est la base des agents IA modernes.

# Cellule 1 — Importer les outils du projet

## Pourquoi cette cellule existe ?

Avant de travailler, Python doit charger les fonctions nécessaires.

C'est comme préparer les outils sur une table avant de bricoler.

Dans cette cellule, on importe :

- `make_client` : pour créer un client LLM.
- `credentials_available` : pour vérifier si on a une vraie clé API.
- `ToolRegistry` : pour ranger les outils disponibles.
- `tool_schema` : pour décrire un outil au modèle.
- `run_agent` : pour exécuter une boucle agentique.
- `safe_calc` : pour faire des calculs simples.

## Ce qu'on doit comprendre

Un agent IA ne travaille pas seul.

Il a besoin :

- d'un modèle ;
- d'outils ;
- d'un programme Python qui orchestre tout.

In [ ]:
from llm_helpers import (
    make_client,
    credentials_available,
    LLMClient,
    MockLLMClient,
    ToolRegistry,
    tool_schema,
    run_agent,
    safe_calc,
)

## Explication de l'output

Cette cellule ne doit normalement rien afficher.

Si elle ne montre aucune erreur, cela veut dire que les imports fonctionnent.

Si tu vois une erreur comme :

```python
ModuleNotFoundError
```

cela veut dire que le fichier `llm_helpers.py` ou certaines dépendances ne sont pas disponibles.

# Cellule 2 — Vérifier si on est en ligne ou hors-ligne

## Pourquoi cette cellule existe ?

Le notebook peut fonctionner de deux façons :

### Mode en ligne

On utilise un vrai modèle LLM avec une clé API.

### Mode hors-ligne

On utilise un faux modèle appelé `MockLLMClient`.

C'est pratique pour apprendre sans payer et sans clé API.

## Image simple

C'est comme apprendre à conduire :

- avec une vraie voiture : mode en ligne ;
- avec un simulateur : mode hors-ligne.

In [ ]:
ONLINE = credentials_available()

print("Mode :", "🌐 EN LIGNE (vrai modèle)" if ONLINE else "⚙️ HORS-LIGNE (mock)")

## Explication de l'output

Tu peux obtenir :

```text
Mode : 🌐 EN LIGNE (vrai modèle)
```

Cela veut dire que le notebook utilise un vrai LLM.

Ou bien :

```text
Mode : ⚙️ HORS-LIGNE (mock)
```

Cela veut dire que le notebook utilise un faux modèle simulé.

Les deux modes sont utiles.

Le mode hors-ligne permet de comprendre le fonctionnement sans dépendre d'une API.

# Cellule 3 — Créer une fonction `demo`

## Pourquoi cette cellule existe ?

On crée une petite fonction pratique appelée `demo`.

Elle permet de créer un client LLM propre à chaque test.

Chaque cellule peut donc être rejouée indépendamment.

## Explication simple

Imagine que chaque exercice utilise une nouvelle feuille blanche.

La fonction `demo` donne une nouvelle feuille blanche au modèle.

In [ ]:
def demo(script=None):
    """Client frais pour une cellule de démonstration."""
    return make_client(offline_script=script, quiet=True)

## Explication de l'output

Cette cellule ne produit pas d'affichage.

Elle crée seulement la fonction `demo`.

Si aucune erreur n'apparaît, la fonction est prête.

# Cellule 4 — Premier test simple avec le LLM

## Objectif

On pose une question très simple :

> En une phrase : qu'est-ce qu'un agent IA ?

## Ce que le modèle doit faire

Il doit répondre avec une phrase courte.

## Explication simple

On vérifie que le modèle sait parler.

In [ ]:
client = demo([
    "Un agent IA perçoit son environnement, raisonne, puis agit pour atteindre un but."
])

rep = client.complete([
    {"role": "user", "content": "En une phrase : qu'est-ce qu'un agent IA ?"}
])

print("→", rep.content)

## Explication de l'output

Exemple d'output :

```text
→ Un agent IA perçoit son environnement, raisonne, puis agit pour atteindre un but.
```

Cela veut dire que le modèle a bien répondu.

Ici, il ne fait aucune action.

Il répond seulement avec du texte.

Donc à ce stade, on a plutôt un comportement de chatbot simple.

# Cellule 5 — Comprendre les rôles : system, user, assistant

## Objectif

Un appel LLM est composé de messages.

Chaque message a un rôle.

| Rôle | Signification |
|---|---|
| `system` | Les règles du jeu |
| `user` | La demande de l'utilisateur |
| `assistant` | La réponse du modèle |

## Explication simple

Le message `system`, c'est comme le professeur.

Le message `user`, c'est comme l'élève qui pose une question.

Le message `assistant`, c'est la réponse.

In [ ]:
def ask(system, user, **kw):
    c = demo([f"[{system[:20]}…] réponse simulée"])
    return c.complete([
        {"role": "system", "content": system},
        {"role": "user", "content": user}
    ], **kw).content

print("PRUDENT :", ask(
    "Tu es un juriste prudent. Réponds en nuançant.",
    "Peut-on déployer un agent autonome en production ?"
))

print("\nDIRECT :", ask(
    "Tu es un ingénieur direct. Réponds en une phrase tranchée.",
    "Peut-on déployer un agent autonome en production ?"
))

## Explication de l'output

On obtient deux réponses différentes.

Pourquoi ?

Parce que le `system prompt` change le comportement du modèle.

### Réponse prudente

Le modèle répond avec nuance.

Il dit par exemple :

> Oui, mais avec des précautions.

### Réponse directe

Le modèle répond plus franchement.

Il peut dire :

> Non, pas sans supervision.

## Ce qu'il faut retenir

Le `system prompt` influence fortement le style, le rôle et les limites du modèle.

# Cellule 6 — Comprendre les tokens et le coût

## Objectif

Chaque appel à un LLM consomme des tokens.

Un token est un petit morceau de texte.

Exemple :

```text
Bonjour Jean
```

peut être découpé en plusieurs tokens.

## Pourquoi c'est important ?

Plus il y a de tokens :

- plus l'appel coûte cher ;
- plus la réponse peut être lente.

C'est une notion importante en FinOps IA.

In [ ]:
c = demo(["Paris est la capitale de la France."])

c.complete([
    {"role": "user", "content": "Quelle est la capitale de la France ?"}
])

print("Dernier appel :", c.last_usage)
print("Cumul         :", c.usage_report())

## Explication de l'output

Exemple :

```text
Dernier appel : {'input_tokens': 9, 'output_tokens': 9}
Cumul         : 1 appel(s) · 9 tok in / 9 tok out · ≈ $0.00003
```

### `input_tokens`

Ce sont les tokens envoyés au modèle.

Donc la question + le contexte.

### `output_tokens`

Ce sont les tokens générés par le modèle.

Donc la réponse.

### Coût

Le coût estimé permet de savoir combien l'appel coûte.

## Ce qu'il faut retenir

Un agent IA peut faire beaucoup d'appels.

Donc il faut surveiller les tokens et le coût dès le début.

# Cellule 7 — Créer un premier outil : obtenir l'heure

## Objectif

Un LLM seul ne connaît pas l'heure actuelle.

On va donc lui donner un outil.

Cet outil s'appelle :

```python
get_current_time
```

## Explication simple

Le LLM est comme un enfant sans montre.

Pour connaître l'heure, il doit demander à quelqu'un qui a une montre.

Ici, la montre, c'est la fonction Python.

In [ ]:
import datetime

def get_current_time(timezone: str = "Europe/Paris") -> str:
    """Renvoie l'heure courante."""
    return f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} ({timezone})"

## Explication de l'output

Cette cellule ne montre rien.

Elle crée seulement une fonction Python.

Cette fonction peut ensuite donner la date et l'heure actuelles.

# Cellule 8 — Décrire l'outil au modèle

## Objectif

Le modèle ne peut pas deviner tout seul que l'outil existe.

On doit lui décrire l'outil.

On lui donne :

- le nom de l'outil ;
- ce qu'il fait ;
- les paramètres qu'il accepte.

## Explication simple

C'est comme donner une fiche technique au modèle.

Exemple :

> Voici un outil. Il s'appelle `get_current_time`. Il sert à donner l'heure.

In [ ]:
time_tool = tool_schema(
    "get_current_time",
    "Donne la date et l'heure courantes.",
    {
        "timezone": {
            "type": "string",
            "description": "ex. Europe/Paris"
        }
    },
)

registry = ToolRegistry()
registry.register(time_tool, get_current_time)

print("Outils disponibles :", registry.names)

## Explication de l'output

Exemple :

```text
Outils disponibles : ['get_current_time']
```

Cela veut dire que l'outil est bien enregistré.

Le modèle peut maintenant savoir qu'il existe un outil pour obtenir l'heure.

# Cellule 9 — Observer la décision du modèle

## Objectif

On pose une question qui nécessite l'outil :

> Quelle heure est-il, précisément ?

Le modèle ne doit pas inventer l'heure.

Il doit décider d'appeler l'outil.

## Point important

À ce moment-là, le modèle ne lance pas encore la fonction.

Il dit seulement :

> Je veux utiliser cet outil.

In [ ]:
client = demo([
    {"tool": "get_current_time", "arguments": {"timezone": "Europe/Paris"}}
])

messages = [
    {"role": "system", "content": "Tu es un assistant. Utilise les outils quand c'est utile."},
    {"role": "user", "content": "Quelle heure est-il, précisément ?"},
]

reply = client.complete(messages, tools=registry.specs)

print("Texte           :", reply.content)
print("Appels d'outils :", reply.tool_calls)

## Explication de l'output

Exemple :

```text
Texte           : None
Appels d'outils : [{'name': 'get_current_time', 'arguments': {...}}]
```

### Pourquoi `Texte : None` ?

Parce que le modèle ne répond pas encore.

Il demande d'abord à utiliser un outil.

### Pourquoi `Appels d'outils` ?

Parce que le modèle a compris qu'il devait appeler `get_current_time`.

## Ce qu'il faut retenir

Le function calling permet au modèle de choisir un outil quand il en a besoin.

# Cellule 10 — Exécuter vraiment l'outil avec `run_agent`

## Objectif

Maintenant, on veut faire la boucle complète :

1. L'utilisateur pose une question.
2. Le modèle décide d'utiliser un outil.
3. Python exécute l'outil.
4. Le résultat revient au modèle.
5. Le modèle donne la réponse finale.

## Explication simple

Le modèle dit :

> J'ai besoin de l'heure.

Python répond :

> Voilà l'heure.

Le modèle dit ensuite :

> Il est actuellement...

In [ ]:
client = demo([
    {"tool": "get_current_time", "arguments": {"timezone": "Europe/Paris"}},
    {"final": "Il est actuellement le créneau affiché ci-dessus ; minuit est dans quelques heures."},
])

messages = [
    {"role": "system", "content": "Tu es un assistant. Utilise les outils quand c'est utile."},
    {"role": "user", "content": "Quelle heure est-il, et dans combien de temps sera-t-il minuit ?"},
]

history = run_agent(client, registry, messages, verbose=True)

## Explication de l'output

Avec `verbose=True`, on voit les étapes.

Exemple :

```text
[étape 1] outil: get_current_time(...)
          ↳ 2026-06-22 17:30:00 (Europe/Paris)

[étape 2] réponse finale → Il est actuellement...
```

### Étape 1

Le modèle demande l'outil.

Python exécute l'outil.

### Étape 2

Le modèle utilise le résultat pour répondre.

## Ce qu'il faut retenir

Un agent IA, ce n'est pas juste un LLM.

C'est une boucle :

```text
LLM → outil → résultat → LLM → réponse finale
```

# Cellule 11 — Le modèle peut aussi ne pas utiliser d'outil

## Objectif

On donne des outils au modèle, mais il n'est pas obligé de les utiliser.

S'il peut répondre directement, il doit répondre directement.

## Exemple

Question :

> En une phrase, qu'est-ce qui distingue un agent d'un simple chatbot ?

Cette question ne demande pas l'heure.

Donc l'outil n'est pas nécessaire.

In [ ]:
client = demo([
    {"final": "Un agent est un système qui perçoit, décide et agit en boucle pour atteindre un but."}
])

reply = client.complete(
    [
        {"role": "system", "content": "Réponds directement ; n'utilise un outil que si nécessaire."},
        {"role": "user", "content": "En une phrase, qu'est-ce qui distingue un agent d'un simple chatbot ?"}
    ],
    tools=registry.specs,
)

print("Tool calls :", reply.tool_calls, "  (attendu : aucun)")
print("Réponse    :", reply.content)

## Explication de l'output

Exemple :

```text
Tool calls : []   (attendu : aucun)
Réponse    : Un agent peut agir, pas seulement répondre.
```

### `Tool calls : []`

Cela veut dire que le modèle n'a appelé aucun outil.

C'est bien.

Il n'avait pas besoin d'outil.

## Ce qu'il faut retenir

Un bon agent ne doit pas utiliser des outils inutilement.

Il doit agir seulement quand c'est utile.

# Cellule 12 — Ajouter un deuxième outil : convertir une monnaie

## Objectif

On ajoute un outil qui convertit un montant avec un taux de change.

Exemple :

```text
250 € × 1.08 = 270 $
```

## Explication simple

On donne une calculatrice spécialisée au modèle.

In [ ]:
def convert_currency(amount: float, rate: float) -> str:
    """Convertit un montant en le multipliant par un taux de change."""
    return f"{amount * rate:.2f}"

registry.register(
    tool_schema(
        "convert_currency",
        "Convertit un montant via un taux de change.",
        {
            "amount": {"type": "number"},
            "rate": {"type": "number"}
        },
        ["amount", "rate"]
    ),
    convert_currency,
)

print("Outils disponibles :", registry.names)

## Explication de l'output

Exemple :

```text
Outils disponibles : ['get_current_time', 'convert_currency']
```

Maintenant, le modèle connaît deux outils :

1. un outil pour l'heure ;
2. un outil pour convertir de l'argent.

# Cellule 13 — Utiliser le deuxième outil

## Objectif

On demande :

> Convertis 250 euros en dollars au taux 1.08.

Le modèle doit utiliser l'outil `convert_currency`.

In [ ]:
client = demo([
    {"tool": "convert_currency", "arguments": {"amount": 250, "rate": 1.08}},
    {"final": "250 € ≈ 270.00 $ au taux 1,08."},
])

run_agent(
    client,
    registry,
    [{"role": "user", "content": "Convertis 250 euros en dollars au taux 1.08."}],
    verbose=True
)

## Explication de l'output

Exemple :

```text
[étape 1] outil: convert_currency({'amount': 250, 'rate': 1.08})
          ↳ 270.00

[étape 2] réponse finale → 250 euros donnent 270.00 dollars.
```

### Étape 1

L'outil calcule :

```text
250 × 1.08 = 270
```

### Étape 2

Le modèle transforme le résultat en phrase humaine.

## Ce qu'il faut retenir

Un agent peut utiliser plusieurs outils selon la demande.

# Cellule 14 — Comprendre la sortie structurée JSON

## Objectif

Un humain aime lire du texte.

Un programme préfère recevoir des données bien rangées.

Exemple humain :

```text
La ville est Lyon, la date est 2026-06-22.
```

Exemple programme :

```json
{
  "ville": "Lyon",
  "date": "2026-06-22",
  "intention": "meteo"
}
```

Le JSON permet de brancher facilement le LLM dans une application.

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "ville": {
            "type": "string"
        },
        "date": {
            "type": "string",
            "description": "AAAA-MM-JJ si présente, sinon ''"
        },
        "intention": {
            "type": "string",
            "enum": ["meteo", "reservation", "autre"]
        },
    },
    "required": ["ville", "intention"],
}

client = demo([
    {
        "tool": "emit",
        "arguments": {
            "ville": "Lyon",
            "date": "2026-06-22",
            "intention": "meteo"
        }
    }
])

phrase = "Quel temps fera-t-il à Lyon le 22 juin 2026 ?"

result = client.complete_structured(
    [{"role": "user", "content": f"Extrait les informations de : '{phrase}'"}],
    schema,
)

print(result)
print("type :", type(result).__name__, "— directement exploitable en Python.")

## Explication de l'output

Exemple :

```text
{'date': '2026-06-22', 'ville': 'Lyon', 'intention': 'meteo'}
type : dict — directement exploitable en Python.
```

### Ce que cela veut dire

Le modèle n'a pas répondu avec une phrase libre.

Il a rempli un dictionnaire Python.

### Pourquoi c'est utile ?

Parce qu'une application peut directement utiliser :

```python
result["ville"]
result["date"]
result["intention"]
```

## Ce qu'il faut retenir

La sortie structurée rend le LLM utilisable dans des applications réelles.

# Cellule 15 — Décrire un agent avec PEAS

## Objectif

Avant de coder un agent, on doit bien définir sa mission.

La méthode PEAS aide à le faire.

| Lettre | Signification |
|---|---|
| P | Performance |
| E | Environment |
| A | Actuators |
| S | Sensors |

## Explication simple

Avant de construire un robot, on doit répondre à quatre questions :

1. Comment savoir s'il réussit ?
2. Où travaille-t-il ?
3. Que peut-il faire ?
4. Que peut-il voir ?

## Exemple : agent assistant de révision

On veut créer un agent qui aide des étudiants à réviser.

### P — Performance

Comment mesure-t-on le succès ?

Réponse :

- taux de bonnes réponses ;
- qualité des QCM ;
- satisfaction de l'étudiant ;
- temps gagné.

### E — Environment

Dans quel monde travaille l'agent ?

Réponse :

- documents de cours ;
- historique de l'étudiant ;
- questions posées.

### A — Actuators

Quelles actions peut-il faire ?

Réponse :

- répondre à des questions ;
- générer des QCM ;
- proposer un planning ;
- citer les sources.

### S — Sensors

Que peut-il percevoir ?

Réponse :

- les messages de l'étudiant ;
- les documents fournis ;
- les résultats des anciens QCM.

# Cellule 16 — Classer l'environnement de l'agent

## Objectif

On décrit maintenant le type d'environnement.

| Propriété | Exemple |
|---|---|
| Observabilité | L'agent voit tout ou seulement une partie ? |
| Déterminisme | Les résultats sont-ils prévisibles ? |
| Épisodicité | Les actions sont-elles indépendantes ? |
| Dynamique | Le monde change-t-il ? |
| Agents | L'agent est-il seul ou avec d'autres agents ? |

## Correction simple

Pour un agent assistant de révision :

- **Partiellement observable** : il ne connaît pas exactement le niveau réel de l'étudiant.
- **Stochastique** : un étudiant peut répondre de manière imprévisible.
- **Séquentiel** : les révisions d'aujourd'hui influencent celles de demain.
- **Dynamique** : les connaissances changent avec le temps.
- **Mono-agent** : dans le cas simple, il n'y a qu'un agent.

# Cellule 17 — Exercice 1 : créer un outil de conversion Celsius → Fahrenheit

## Objectif

Tu dois créer un outil :

```python
temperature_convert(celsius)
```

Il convertit une température Celsius en Fahrenheit.

## Formule

```text
Fahrenheit = Celsius × 9/5 + 32
```

Exemple :

```text
25°C = 77°F
```

In [ ]:
def temperature_convert(celsius: float) -> str:
    return f"{celsius * 9/5 + 32:.1f} °F"

registry.register(
    tool_schema(
        "temperature_convert",
        "Convertit des degrés Celsius en degrés Fahrenheit.",
        {
            "celsius": {"type": "number"}
        },
        ["celsius"]
    ),
    temperature_convert,
)

print("Outils disponibles :", registry.names)

## Explication de l'output

Exemple :

```text
Outils disponibles : ['get_current_time', 'convert_currency', 'temperature_convert']
```

Cela veut dire que le nouvel outil est bien ajouté.

Le modèle peut maintenant convertir des températures.

# Cellule 18 — Exercice 1 corrigé : utiliser deux outils ensemble

## Objectif

On demande au modèle :

> Quelle heure est-il, et combien font 25°C en Fahrenheit ?

Le modèle doit utiliser :

1. `get_current_time`
2. `temperature_convert`

## Ce qu'on veut observer

L'agent peut enchaîner plusieurs outils.

In [ ]:
client = demo([
    {"tool": "get_current_time", "arguments": {"timezone": "Europe/Paris"}},
    {"tool": "temperature_convert", "arguments": {"celsius": 25}},
    {"final": "Il est l'heure indiquée ; 25 °C = 77.0 °F."},
])

run_agent(
    client,
    registry,
    [{"role": "user", "content": "Quelle heure est-il, et combien font 25°C en Fahrenheit ?"}],
    verbose=True
)

## Explication de l'output

Exemple :

```text
[étape 1] outil: get_current_time(...)
          ↳ 2026-06-22 17:40:00 (Europe/Paris)

[étape 2] outil: temperature_convert({'celsius': 25})
          ↳ 77.0 °F

[étape 3] réponse finale → Il est l'heure indiquée ; 25 °C = 77.0 °F.
```

## Ce qu'il faut retenir

Un agent peut composer plusieurs outils pour répondre à une question plus complexe.

# Cellule 19 — Exercice 2 : extraire des informations d'un email

## Objectif

On veut transformer un email en données structurées.

On veut extraire :

- l'expéditeur ;
- le niveau d'urgence ;
- l'action requise.

## Pourquoi c'est utile ?

Dans une entreprise, cela peut servir à créer automatiquement un ticket support.

In [ ]:
email_schema = {
    "type": "object",
    "properties": {
        "expediteur": {
            "type": "string"
        },
        "urgence": {
            "type": "string",
            "enum": ["faible", "moyenne", "haute"]
        },
        "action_requise": {
            "type": "string"
        },
    },
    "required": ["urgence", "action_requise"],
}

texte = "De: dsi@aivancity.fr — Le serveur de prod est down, merci d'intervenir immédiatement."

client = demo([
    {
        "tool": "emit",
        "arguments": {
            "expediteur": "dsi@aivancity.fr",
            "urgence": "haute",
            "action_requise": "Rétablir le serveur de production"
        }
    }
])

result = client.complete_structured(
    [{"role": "user", "content": f"Extrait les infos de cet email : {texte}"}],
    email_schema
)

print(result)

## Explication de l'output

Exemple :

```python
{
    'expediteur': 'dsi@aivancity.fr',
    'urgence': 'haute',
    'action_requise': 'Rétablir le serveur de production'
}
```

### Pourquoi urgence = haute ?

Parce que le texte dit :

```text
Le serveur de prod est down
```

et :

```text
intervenir immédiatement
```

Cela indique une situation critique.

## Ce qu'il faut retenir

Un LLM peut lire un texte non structuré et le transformer en données propres.

# Récapitulatif final

Dans ce notebook, tu as appris :

## 1. Un appel LLM

C'est une conversation envoyée à un modèle.

Elle contient des messages :

- `system`
- `user`
- `assistant`

## 2. Le system prompt

Il donne les règles au modèle.

## 3. La température

Elle contrôle le niveau de créativité.

Pour les agents fiables, on utilise souvent une température basse.

## 4. Les tokens

Ils permettent de mesurer le coût et la taille des appels.

## 5. Le function calling

Le modèle peut demander à utiliser un outil.

## 6. Les outils

Un outil est une fonction Python que le modèle peut appeler.

## 7. La boucle agentique

```text
Utilisateur → LLM → outil → résultat → LLM → réponse finale
```

## 8. Le JSON structuré

Il permet d'obtenir une réponse propre et exploitable par un programme.

## 9. PEAS

PEAS sert à définir clairement la mission d'un agent.

---

# Phrase à retenir

Un agent IA, c'est un LLM + des outils + une boucle de décision.